# Scraping WDumps data
<a href="https://colab.research.google.com/github/wmde/WDumps-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages


## Preparing the workspace

### Importing libraries

In [ ]:
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from operator import itemgetter

from requests_ratelimiter import SQLiteBucket

from wdumps_scraper import (
    CachedLimiterSession,
    CacheDuration,
    ClientError,
    Scraper,
    WDumperClient,
    parsing,
    rendering,
)

### Global definitions

In [ ]:
session = CachedLimiterSession(
    "scraper_cache",
    backend="sqlite",
    expire_after=None,
    user_agent="wdumps_scraper/0.1 (https://github.com/wmde/WDumps-scraper)",
    per_second=10,
    bucket_class=SQLiteBucket,
    bucket_kwargs={
        "path": "ratelimiter.sqlite",
        "isolation_level": "EXCLUSIVE",
        "check_same_thread": False,
    },
)

scraper = Scraper(session)
dumper_client = WDumperClient(session)


def scrape_recent_dumps():
    url = "https://wdumps.toolforge.org/dumps"
    html = scraper.get(url)
    return parsing.extract_last_id(html)


def scrape_dump(dump_id):
    cache_duration = (
        CacheDuration.INDEFINITE if dump_id < last_id - 10 else CacheDuration.LOW
    )

    try:
        data = dumper_client.get_dump(dump_id, cache_duration)
        includes = rendering.render_includes(data["spec"])
        languages = rendering.render_languages(data["spec"])
        return True, {
            "url": data["url"],
            "id": data["id"],
            "name": data["title"],
            "includes": includes,
            "languages": languages,
        }
    except ClientError as e:
        return False, {"id": dump_id, "error": str(e)}

## Extract latest dump ID

In [ ]:
last_id = scrape_recent_dumps()
print(last_id)

## Extract IDs, names and inclusion information
Save these in a dictionary with the URLs.

In [ ]:
dumps = []
skipped = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {}

    for i in range(300, 0, -1):
        future = executor.submit(scrape_dump, i)
        futures[future] = i

    for future in as_completed(futures):
        id_ = futures[future]
        try:
            success, result = future.result()
            if success:
                dumps.append(result)
            else:
                skipped.append(result)
        except Exception as e:
            skipped.append({"id": id_, "error": str(e)})


print(f"Dumps scraped: {len(dumps)}")
print(f"Skipped: {len(skipped)}")

## Save data to csv

In [ ]:
sorted_dumps = sorted(dumps, key=itemgetter("id"), reverse=True)

with open("wdumps-data.csv", "w", newline="") as csvfile:
    fieldnames = ["id", "name", "includes", "languages", "url"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for dump in sorted_dumps:
        writer.writerow(dump)

In [ ]:
print(skipped[40])